In [ ]:
# Import Libraries
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

import random

In [ ]:
# Load Dataset

df = pd.read_csv("Cleaned-Recipies-dataset.csv")

df.head()

,recipe_id,name,cuisine,category,ingredients,steps,calories,veg,flavor_profile,season,suitable_for_diabetes,suitable_for_bp,suitable_for_heart_condition,ingredients_length,steps_length,health_tags
0,1,garlic naan,Pakistani,bread,"500g wheat/all-purpose flour, 1 cup water, 2 t...",1. sieve the flour and add salt and water grad...,352,1,savory,all,1,NaN,0,4,11,diabetic
1,2,tandoori naan,Pakistani,bread,"500g wheat/all-purpose flour, 1 cup water, 2 t...",1. sieve the flour and add salt and water grad...,173,1,savory,all,1,1.0,1,4,11,"diabetic,bp,heart"
2,3,dahi bhallay,Pakistani,snack,"flour/besan, lemon, oil for frying, spices","1. in a large bowl, mix the base ingredients w...",310,1,savory,all,1,1.0,1,4,13,"diabetic,bp,heart"
3,4,bun kebab,Pakistani,snack,"flour/besan, potato, oil for frying, spices","1. in a large bowl, mix the base ingredients w...",256,0,spicy,all,1,1.0,0,4,13,"diabetic,bp"
4,5,chicken biryani,Pakistani,rice,"500g basmati rice, sugar, whole spices, ghee",1. wash the rice multiple times until the wate...,790,0,spicy,all,0,NaN,0,4,13,NaN


In [ ]:
# Check Important Columns

print(df.columns)

Index(['recipe_id', 'name', 'cuisine', 'category', 'ingredients', 'steps',
       'calories', 'veg', 'flavor_profile', 'season', 'suitable_for_diabetes',
       'suitable_for_bp', 'suitable_for_heart_condition', 'ingredients_length',
       'steps_length', 'health_tags'],
      dtype='object')


In [ ]:
# Combine Features (TF-IDF )

print(df.columns)
df['combined'] = df['ingredients'] + " " + df['steps'] + " " + df['name']

Index(['recipe_id', 'name', 'cuisine', 'category', 'ingredients', 'steps',
       'calories', 'veg', 'flavor_profile', 'season', 'suitable_for_diabetes',
       'suitable_for_bp', 'suitable_for_heart_condition', 'ingredients_length',
       'steps_length', 'health_tags', 'combined'],
      dtype='object')


In [ ]:
# TF-IDF Apply

from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(stop_words='english')

tfidf_matrix = tfidf.fit_transform(df['combined'])

print("\n TF-IDF Applied")
print("Shape of Matrix:", tfidf_matrix.shape)

print("\nSample Features:")
print(tfidf.get_feature_names_out()[:20])


 TF-IDF Applied
Shape of Matrix: (7000, 519)

Sample Features:
['10' '100' '101' '102' '103' '104' '105' '106' '107' '108' '109' '11'
 '110' '111' '112' '113' '114' '115' '116' '117']


In [ ]:
# Recommendation Function

from sklearn.metrics.pairwise import cosine_similarity

def recommend_meals(user_input):
    
    user_vec = tfidf.transform([user_input])
    
    similarity = cosine_similarity(user_vec, tfidf_matrix)
    
    scores = list(enumerate(similarity[0]))
    
    scores = sorted(scores, key=lambda x: x[1], reverse=True)
    
    top_indices = [i[0] for i in scores[:10]]
    
    return df.iloc[top_indices]
    
# Output test
    
test_input = "chicken spicy low calorie"

result = recommend_meals(test_input)

print("\n Top Recommendations:\n")

for i, row in result.iterrows():
    print("Recipe:", row['name'])
    print("Category:", row['category'])
    print("----------------------")


 Top Recommendations:

Recipe: spicy spaghetti 2
Category: pasta
----------------------
Recipe: spicy spaghetti 7
Category: pasta
----------------------
Recipe: spicy spaghetti 9
Category: pasta
----------------------
Recipe: spicy spaghetti 11
Category: pasta
----------------------
Recipe: spicy spaghetti 23
Category: pasta
----------------------
Recipe: spicy spaghetti 29
Category: pasta
----------------------
Recipe: spicy spaghetti 32
Category: pasta
----------------------
Recipe: spicy spaghetti 51
Category: pasta
----------------------
Recipe: spicy spaghetti 52
Category: pasta
----------------------
Recipe: spicy spaghetti 53
Category: pasta
----------------------


In [ ]:
# 7 Day Meal Planner Generate 

def generate_meal_plan(user_input):
    
    recommendations = recommend_meals(user_input)
    
    meal_plan = {}
    
    for day in range(1, 8):
        
        breakfast = recommendations[recommendations['category'].str.lower() == 'breakfast']
        lunch = recommendations[recommendations['category'].str.lower() == 'lunch']
        dinner = recommendations[recommendations['category'].str.lower() == 'dinner']
        
        # fallback
        if breakfast.empty:
            breakfast = recommendations
        if lunch.empty:
            lunch = recommendations
        if dinner.empty:
            dinner = recommendations
        
        meal_plan[f"Day {day}"] = {
            "Breakfast": breakfast.sample(1)['name'].values[0],
            "Lunch": lunch.sample(1)['name'].values[0],
            "Dinner": dinner.sample(1)['name'].values[0]
        }
    
    return meal_plan

In [ ]:
# Final Input and Output 

user_input = input("\nEnter your preference (e.g., chicken, spicy, low calorie): ")

meal_plan = generate_meal_plan(user_input)

print("\n Your 7-Day Meal Plan:\n")

for day, meals in meal_plan.items():
    print(day)
    print(" Breakfast:", meals["Breakfast"])
    print(" Lunch:", meals["Lunch"])
    print(" Dinner:", meals["Dinner"])
    print("--------------------------")

# Confirmation
confirm = input("\nDo you like this plan? (yes/no): ")

if confirm.lower() == "yes":
    print("\n Meal Plan Confirmed!")
else:
    print("\n Generating New Plan...\n")
    
    meal_plan = generate_meal_plan(user_input)
    
    for day, meals in meal_plan.items():
        print(day)
        print(" Breakfast:", meals["Breakfast"])
        print(" Lunch:", meals["Lunch"])
        print(" Dinner:", meals["Dinner"])
        print("--------------------------")


Enter your preference (e.g., chicken, spicy, low calorie):  chicken



 Your 7-Day Meal Plan:

Day 1
 Breakfast: chicken haleem
 Lunch: chicken haleem
 Dinner: chicken paaya
--------------------------
Day 2
 Breakfast: chicken tikka
 Lunch: chicken handi
 Dinner: chicken saag
--------------------------
Day 3
 Breakfast: chicken karahi
 Lunch: chicken karahi
 Dinner: chicken haleem
--------------------------
Day 4
 Breakfast: chicken saag
 Lunch: chicken tikka
 Dinner: chicken karahi
--------------------------
Day 5
 Breakfast: chicken daal
 Lunch: chicken karahi
 Dinner: chicken tikka
--------------------------
Day 6
 Breakfast: chicken karahi
 Lunch: chicken haleem
 Dinner: chicken handi
--------------------------
Day 7
 Breakfast: chicken tikka
 Lunch: chicken karahi
 Dinner: chicken tikka
--------------------------



Do you like this plan? (yes/no):  no



 Generating New Plan...

Day 1
 Breakfast: chicken paaya
 Lunch: chicken karahi
 Dinner: chicken saag
--------------------------
Day 2
 Breakfast: chicken handi
 Lunch: chicken haleem
 Dinner: chicken saag
--------------------------
Day 3
 Breakfast: chicken handi
 Lunch: chicken paaya
 Dinner: chicken tikka
--------------------------
Day 4
 Breakfast: chicken karahi
 Lunch: chicken sajji
 Dinner: chicken haleem
--------------------------
Day 5
 Breakfast: chicken haleem
 Lunch: chicken daal
 Dinner: chicken paaya
--------------------------
Day 6
 Breakfast: chicken handi
 Lunch: chicken daal
 Dinner: chicken haleem
--------------------------
Day 7
 Breakfast: chicken sajji
 Lunch: chicken haleem
 Dinner: chicken paaya
--------------------------
